In [ ]:
import os
import glob
import json
import pickle
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd

BASE_DIR = os.path.dirname(os.path.abspath("d:\git_repos\capstone\graph-transformer\preprocess.py"))
ROAD_ROOT = os.path.join(BASE_DIR, "road", "road")

AMBIENT_DIR = os.path.join(ROAD_ROOT, "signal_extractions", "ambient")
ATTACK_DIR = os.path.join(ROAD_ROOT, "signal_extractions", "attacks")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

WINDOW_SIZE_SEC = 2.0
STRIDE_SEC_BENIGN = 1.0
STRIDE_SEC_ATTACK = 2.0
MIN_MESSAGES_PER_WINDOW = 5

KEEP_RAW_WINDOW_MESSAGES = True


In [22]:
@dataclass
class CaptureFile:
    path: str
    capture_name: str
    is_attack: bool
    metadata: dict = field(default_factory=dict)

In [23]:
@dataclass
class WindowRecord:

    capture_name: str
    window_start: float
    window_end: float
    messages: Optional[pd.DataFrame] = None
    label: int = 0
    frac_attack_messages: float = 0.0

In [24]:
def discover_captures(ambient_dir: str, attack_dir: str) -> list[CaptureFile]:

    captures = []

    for csv_path in sorted(glob.glob(os.path.join(ambient_dir, "*.csv"))):
        filename = os.path.basename(csv_path)
        if not filename.startswith("ambient_"):
            continue
        name = os.path.splitext(filename)[0]
        captures.append(CaptureFile(
            path=csv_path,
            capture_name=name,
            is_attack=False,
            metadata=_load_metadata_if_exists(csv_path),
        ))

    for csv_path in sorted(glob.glob(os.path.join(attack_dir, "*.csv"))):
        name = os.path.splitext(os.path.basename(csv_path))[0]
        captures.append(CaptureFile(
            path=csv_path,
            capture_name=name,
            is_attack=True,
            metadata=_load_metadata_if_exists(csv_path),
        ))

    if not captures:
        raise FileNotFoundError(
            f"No CSV captures found under {ambient_dir} or {attack_dir}. "
            f"Check ROAD_ROOT / AMBIENT_DIR / ATTACK_DIR paths."
        )

    return captures

In [25]:
def _load_metadata_if_exists(csv_path: str) -> dict:
   
    candidates = [
        os.path.join(os.path.dirname(csv_path), "metadata.json"),
        csv_path.replace(".csv", "_metadata.json"),
    ]

    for candidate in candidates:
        if os.path.exists(candidate):
            with open(candidate, "r") as f:
                return json.load(f)
    return {}

In [26]:
def load_capture(capture: CaptureFile) -> pd.DataFrame:

    df = pd.read_csv(capture.path)

    expected_cols = {"Label", "ID", "Time"}
    missing = expected_cols - set(df.columns)
    if missing:
        raise ValueError(
            f"{capture.path} is missing expected columns: {missing}. "
            f"Confirm this is a ROAD signal-translated CSV."
        )

    df["Label"] = df["Label"].astype(int)
    df["ID"] = df["ID"].astype(str)
    df["Time"] = df["Time"].astype(float)
    df = df.sort_values("Time").reset_index(drop=True)

    df["Time"] = df["Time"] - df["Time"].min()

    return df

In [27]:
def window_capture(df: pd.DataFrame, capture_name: str, is_attack: bool, keep_messages: bool = KEEP_RAW_WINDOW_MESSAGES) -> list[WindowRecord]:
    stride = STRIDE_SEC_BENIGN if not is_attack else STRIDE_SEC_ATTACK
    total_duration = df["Time"].max()

    windows = []
    window_start = 0.0

    while window_start < total_duration:
        window_end = window_start + WINDOW_SIZE_SEC

        mask = (df["Time"] >= window_start) & (df["Time"] < window_end)
        window_df = df.loc[mask]

        if len(window_df) >= MIN_MESSAGES_PER_WINDOW:
            frac_attack = float(window_df["Label"].mean())
            window_label = int(window_df["Label"].max())

            windows.append(WindowRecord(
                capture_name=capture_name,
                window_start=window_start,
                window_end=window_end,
                messages=window_df.reset_index(drop=True) if keep_messages else None,
                label=window_label,
                frac_attack_messages=frac_attack,
            ))

        window_start += stride

    return windows

In [28]:
def build_dataset(keep_messages: bool = KEEP_RAW_WINDOW_MESSAGES) -> list[WindowRecord]:
    captures = discover_captures(AMBIENT_DIR, ATTACK_DIR)
    print(f"Discovered {len(captures)} captures "
          f"({sum(not c.is_attack for c in captures)} ambient, "
          f"{sum(c.is_attack for c in captures)} attack).")

    all_windows: list[WindowRecord] = []

    for capture in captures:
        print(f"  Processing {capture.capture_name} "
              f"({'attack' if capture.is_attack else 'ambient'}) ...")

        df = load_capture(capture)
        windows = window_capture(df, capture.capture_name, capture.is_attack, keep_messages=keep_messages)
        all_windows.extend(windows)

        print(f"    -> {len(df)} messages, {len(windows)} windows kept "
              f"(min {MIN_MESSAGES_PER_WINDOW} msgs/window)")

    n_benign = sum(w.label == 0 for w in all_windows)
    n_attack = sum(w.label == 1 for w in all_windows)
    print(f"\nTotal windows: {len(all_windows)} "
          f"({n_benign} benign, {n_attack} attack)")

    return all_windows

In [29]:
def save_dataset(windows: list[WindowRecord], output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, "road_windowed.pkl")
    with open(out_path, "wb") as f:
        pickle.dump(windows, f)
    print(f"Saved {len(windows)} windows to {out_path}")

    meta_path = os.path.join(output_dir, "road_windowed_metadata.json")
    metadata = {
        "window_duration_sec": WINDOW_SIZE_SEC,
        "benign_stride_sec": STRIDE_SEC_BENIGN,
        "attack_stride_sec": STRIDE_SEC_ATTACK,
        "min_messages_per_window": MIN_MESSAGES_PER_WINDOW,
        "total_windows": len(windows),
        "benign_windows": sum(w.label == 0 for w in windows),
        "attack_windows": sum(w.label == 1 for w in windows),
    }
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"Saved windowing metadata to {meta_path}")

In [34]:
if __name__ == "__main__":
    windows = build_dataset()
    save_dataset(windows, OUTPUT_DIR)

Discovered 29 captures (12 ambient, 17 attack).
  Processing ambient_dyno_drive_basic_long (ambient) ...
    -> 2802432 messages, 1222 windows kept (min 5 msgs/window)
  Processing ambient_dyno_drive_basic_short (ambient) ...
    -> 996482 messages, 445 windows kept (min 5 msgs/window)
  Processing ambient_dyno_drive_benign_anomaly (ambient) ...
    -> 720929 messages, 436 windows kept (min 5 msgs/window)
  Processing ambient_dyno_drive_extended_long (ambient) ...
    -> 1335049 messages, 645 windows kept (min 5 msgs/window)
  Processing ambient_dyno_drive_extended_short (ambient) ...
    -> 741572 messages, 360 windows kept (min 5 msgs/window)
  Processing ambient_dyno_drive_radio_infotainment (ambient) ...
    -> 874018 messages, 391 windows kept (min 5 msgs/window)
  Processing ambient_dyno_drive_winter (ambient) ...
    -> 106939 messages, 48 windows kept (min 5 msgs/window)
  Processing ambient_dyno_exercise_all_bits (ambient) ...
    -> 4136020 messages, 2101 windows kept (min 5 